# LeetCode #61 — Rotate List: xoay Linked List bằng cách nối vòng rồi cắt

**Mức độ:** Medium · **Chủ đề:** Linked List, Two Pointers

Skeleton trên LeetCode:

```python
# Definition for singly-linked list.
# class ListNode:
#     def __init__(self, val=0, next=None):
#         self.val = val
#         self.next = next
class Solution:
    def rotateRight(self, head: Optional[ListNode], k: int) -> Optional[ListNode]:
```

Mục tiêu của notebook:

1. Thấy vì sao xoay **từng bước** thất bại khi `k` tới `2 * 10^9`.
2. Rút gọn `k` bằng **`k % độ_dài`**.
3. Kỹ thuật **nối thành vòng rồi cắt** đúng chỗ — chỉ sửa **hai** mũi tên.
4. Phiên bản dùng **hai con trỏ cách nhau `k` bước** (giống bài #19).

# 1. Bài toán là gì?

Cho `head` của Linked List, **xoay sang phải** `k` vị trí: mỗi lần xoay, nút cuối chuyển lên đầu.

**Constraints (theo đề):**

- Số nút trong `[0, 500]`
- `-100 <= Node.val <= 100`
- `0 <= k <= 2 * 10^9`

---

## Ví dụ

```text
[1, 2, 3, 4, 5], k = 2  ->  [4, 5, 1, 2, 3]
[0, 1, 2],       k = 4  ->  [2, 0, 1]
```

Ví dụ 2 từng bước: `[0,1,2] → [2,0,1] → [1,2,0] → [0,1,2] → [2,0,1]`. Xoay 3 lần (bằng độ dài) quay về như cũ!

In [1]:
from typing import Optional, List


class ListNode:
    def __init__(self, val=0, next=None):
        self.val = val
        self.next = next


def build_list(values: List[int]) -> Optional[ListNode]:
    dummy = ListNode()
    tail = dummy
    for v in values:
        tail.next = ListNode(v)
        tail = tail.next
    return dummy.next


def to_list(head: Optional[ListNode]) -> List[int]:
    out = []
    while head:
        out.append(head.val)
        head = head.next
    return out

print(to_list(build_list([1, 2, 3, 4, 5])))

[1, 2, 3, 4, 5]


# 2. Kiến thức nền cần nắm trước

Để hiểu trọn lời giải bài này, bạn cần nắm chắc các kiến thức dưới đây. Mỗi phần trình bày: khái niệm là gì, hình minh họa, code mẫu **chạy được**, độ phức tạp và lỗi hay gặp. Phần nào đã vững, bạn có thể đọc lướt.

1. Danh sách liên kết đơn (singly linked list)

Cuối mục có phần **2.2 Áp dụng vào bài này** nối các kiến thức trên với lời giải.

## 2.1 Danh sách liên kết đơn (singly linked list)

### Danh sách liên kết là gì?

Mảng (`list` của Python) lưu các phần tử **liền nhau** trong bộ nhớ, nên truy cập `a[i]` tức thì. Danh sách liên kết thì khác: mỗi phần tử là một **nút** (node) riêng lẻ, chứa **giá trị** và **con trỏ** `next` chỉ tới nút kế tiếp. Nút cuối có `next = None`.

```text
head
 │
 ▼
┌───┬───┐   ┌───┬───┐   ┌───┬────┐
│ 1 │ ●─┼──►│ 2 │ ●─┼──►│ 3 │None│
└───┴───┘   └───┴───┘   └───┴────┘
 val next
```

Ta chỉ giữ con trỏ tới nút đầu (`head`). Muốn tới nút thứ `k` phải đi qua `k` bước — **không** có truy cập ngẫu nhiên.

| | Mảng | Danh sách liên kết |
|---|---|---|
| Lấy phần tử thứ `i` | `O(1)` | `O(i)` |
| Chèn/xóa ở đầu | `O(n)` | `O(1)` |
| Chèn/xóa sau một nút đã có trong tay | `O(n)` | `O(1)` |

In [2]:
class ListNode:
    def __init__(self, val=0, next=None):
        self.val = val
        self.next = next

# Tạo bằng tay: 1 -> 2 -> 3
c = ListNode(3)
b = ListNode(2, c)
a = ListNode(1, b)
head = a
print(head.val, head.next.val, head.next.next.val, head.next.next.next)

1 2 3 None


### Duyệt danh sách và hai hàm tiện ích

Mẫu duyệt cơ bản: con trỏ `cur` bắt đầu ở `head`, mỗi bước `cur = cur.next`, dừng khi `cur is None`.

In [3]:
def tao_danh_sach(values):
    """[1, 2, 3] -> 1 -> 2 -> 3, trả về head."""
    dummy = tail = ListNode()          # nút giả để khỏi xử lý riêng nút đầu
    for v in values:
        tail.next = ListNode(v)
        tail = tail.next
    return dummy.next

def ra_list(head):
    out, cur = [], head
    while cur:                          # duyệt tới khi hết
        out.append(cur.val)
        cur = cur.next
    return out

h = tao_danh_sach([4, 5, 6, 7])
print("ra_list:", ra_list(h))
dem, cur = 0, h
while cur:
    dem += 1
    cur = cur.next
print("độ dài:", dem)

ra_list: [4, 5, 6, 7]
độ dài: 4


### Chèn và xóa: chỉ cần đổi con trỏ

Chèn nút `x` **sau** nút `p`: `x.next = p.next; p.next = x` (thứ tự quan trọng — làm ngược sẽ mất phần sau).

Xóa nút **sau** `p`: `p.next = p.next.next`.

```text
Chèn x sau p:   p ──► q        ⟹   p ──► x ──► q
Xóa sau p:      p ──► q ──► r  ⟹   p ──────────► r
```

Muốn xóa một nút, ta cần nút **đứng trước** nó. Nếu nút cần xóa là `head` thì không có nút trước → dùng **nút giả (dummy)** đặt trước `head`.

In [4]:
h = tao_danh_sach([1, 2, 4])
p = h.next                              # nút 2
x = ListNode(3)
x.next = p.next                         # 3 -> 4
p.next = x                              # 2 -> 3
print("sau khi chèn 3:", ra_list(h))

def xoa_gia_tri(head, target):
    dummy = ListNode(0, head)           # dummy đứng trước head
    prev = dummy
    while prev.next:
        if prev.next.val == target:
            prev.next = prev.next.next  # bỏ qua nút cần xóa
        else:
            prev = prev.next
    return dummy.next                   # head mới (có thể đã đổi)

print("xóa mọi số 1:", ra_list(xoa_gia_tri(tao_danh_sach([1, 1, 2, 1, 3]), 1)))

sau khi chèn 3: [1, 2, 3, 4]
xóa mọi số 1: [2, 3]


### Đảo ngược danh sách — kỹ thuật ba con trỏ

Đi dọc danh sách, lật mũi tên của từng nút cho trỏ về nút **trước** nó. Cần lưu `nxt` trước khi lật, vì sau khi lật ta mất đường đi tiếp.

```text
prev  cur   nxt
None   1 ──► 2 ──► 3          bước 1: 1.next = None
None ◄─1     2 ──► 3          bước 2: 2.next = 1
None ◄─1 ◄─ 2     3          bước 3: 3.next = 2  → head mới = 3
```

In [5]:
def dao_nguoc(head):
    prev, cur = None, head
    while cur:
        nxt = cur.next      # 1) lưu đường đi tiếp
        cur.next = prev     # 2) lật mũi tên
        prev = cur          # 3) tiến prev
        cur = nxt           # 4) tiến cur
    return prev             # prev là head mới

print("đảo [1..5]:", ra_list(dao_nguoc(tao_danh_sach([1, 2, 3, 4, 5]))))

đảo [1..5]: [5, 4, 3, 2, 1]


### Con trỏ nhanh / chậm (fast & slow)

`slow` đi 1 bước, `fast` đi 2 bước mỗi vòng.

- Khi `fast` tới cuối, `slow` ở **giữa** danh sách.
- Nếu danh sách có **chu trình**, `fast` sẽ đuổi kịp `slow` (thuật toán Floyd).

In [6]:
def nut_giua(head):
    slow = fast = head
    while fast and fast.next:
        slow = slow.next
        fast = fast.next.next
    return slow

print("giữa [1..5]:", nut_giua(tao_danh_sach([1, 2, 3, 4, 5])).val,
      "| giữa [1..6]:", nut_giua(tao_danh_sach([1, 2, 3, 4, 5, 6])).val, "(nút giữa thứ hai)")

def co_chu_trinh(head):
    slow = fast = head
    while fast and fast.next:
        slow, fast = slow.next, fast.next.next
        if slow is fast:
            return True
    return False

v = tao_danh_sach([1, 2, 3, 4])
v.next.next.next.next = v.next          # 4 -> 2 tạo chu trình
print("có chu trình?", co_chu_trinh(v), "| danh sách thường:", co_chu_trinh(tao_danh_sach([1, 2])))

giữa [1..5]: 3 | giữa [1..6]: 4 (nút giữa thứ hai)
có chu trình? True | danh sách thường: False


### Lỗi hay gặp với danh sách liên kết

- Truy cập `cur.next.val` khi `cur.next` là `None` → `AttributeError`. Luôn kiểm tra `cur and cur.next`.
- Đổi `cur.next` **trước** khi lưu nút kế → mất phần còn lại của danh sách.
- Quên trường hợp danh sách rỗng (`head is None`) hoặc chỉ một nút.
- Quên dùng dummy khi `head` có thể bị xóa/thay → phải viết nhánh riêng, dễ sai.
- In danh sách có chu trình bằng vòng `while cur` → lặp vô hạn.

## 2.2 Áp dụng vào bài này

Xoay **danh sách liên kết**: đếm độ dài `n` và tìm nút cuối, **nối vòng** (`tail.next = head`), rồi đi `n − k % n` bước để tới chỗ **cắt** — nút đó thành đuôi mới (`next = None`), nút sau nó là `head` mới.

# 3. Cách dễ nghĩ: xoay từng bước

Mỗi bước: tìm nút **áp chót**, tách nút cuối ra, đặt lên đầu. Mỗi bước `O(n)`, `k` bước → `O(k · n)`.

In [7]:
def rotate_step_by_step(head: Optional[ListNode], k: int) -> Optional[ListNode]:
    if not head or not head.next:
        return head
    for _ in range(k):
        prev, cur = None, head
        while cur.next:                 # tìm nút cuối và nút áp chót
            prev, cur = cur, cur.next
        prev.next = None                # tách nút cuối
        cur.next = head                 # đặt lên đầu
        head = cur
    return head

print(to_list(rotate_step_by_step(build_list([1, 2, 3, 4, 5]), 2)))
print(to_list(rotate_step_by_step(build_list([0, 1, 2]), 4)))

[4, 5, 1, 2, 3]
[2, 0, 1]


Với `k = 2 * 10^9` và `n = 500`: `10^12` thao tác — không thể chạy.

# 4. Quan sát 1: xoay `n` lần quay về như cũ

Vì vậy chỉ cần xoay `k % n` lần.

In [8]:
for k in [0, 2, 5, 7, 2 * 10**9]:
    print(f"k = {k:>10}: k % 5 = {k % 5}")

k =          0: k % 5 = 0
k =          2: k % 5 = 2
k =          5: k % 5 = 0
k =          7: k % 5 = 2
k = 2000000000: k % 5 = 0


# 5. Quan sát 2: xoay `k` lần = cắt ở đúng một chỗ

Xoay `[1, 2, 3, 4, 5]` sang phải `k = 2`:

```text
trước:  1 -> 2 -> 3 -> 4 -> 5
sau:    4 -> 5 -> 1 -> 2 -> 3
```

Đây chỉ là **lấy `k` nút cuối đưa lên đầu**. Thứ tự bên trong mỗi phần giữ nguyên. Chỉ cần sửa:

1. Nút cuối cũ (`5`) trỏ về head cũ (`1`) — **nối vòng**.
2. Nút thứ `n - k` (`3`) trỏ về `None` — **cắt**.
3. Head mới là nút thứ `n - k + 1` (`4`).

```text
nối vòng:  1 -> 2 -> 3 -> 4 -> 5 -+
           ^-----------------------+
cắt sau 3: 4 -> 5 -> 1 -> 2 -> 3 -> None
```

# 6. Lời giải chính

1. Đếm độ dài `n` và tìm nút cuối `tail` (một lần duyệt).
2. `k %= n`; nếu `k == 0` trả `head` luôn.
3. Nối vòng: `tail.next = head`.
4. Từ nút cuối `tail`, đi `n - k` bước → tới nút thứ `n - k` (đánh số từ 1): đó là `new_tail`.
5. `new_head = new_tail.next`, `new_tail.next = None`.

Bước 4: đứng ở `tail` (vị trí 0 trên vòng, ngay trước `head`), đi `n - k` bước sẽ tới nút thứ `n - k`.

In [9]:
def rotate_right(head: Optional[ListNode], k: int) -> Optional[ListNode]:
    if not head or not head.next:
        return head

    n, tail = 1, head
    while tail.next:
        tail = tail.next
        n += 1

    k %= n
    if k == 0:
        return head

    tail.next = head                    # nối vòng
    new_tail = tail
    for _ in range(n - k):
        new_tail = new_tail.next
    new_head = new_tail.next
    new_tail.next = None                # cắt
    return new_head

print(to_list(rotate_right(build_list([1, 2, 3, 4, 5]), 2)))
print(to_list(rotate_right(build_list([0, 1, 2]), 4)))
print(to_list(rotate_right(build_list([]), 3)))

[4, 5, 1, 2, 3]
[2, 0, 1]
[]


# 7. Mô phỏng từng bước

In [10]:
def trace_rotate(values: List[int], k: int):
    head = build_list(values)
    if not head or not head.next:
        print("0 hoặc 1 nút -> giữ nguyên", values)
        return
    n, tail = 1, head
    while tail.next:
        tail = tail.next
        n += 1
    print(f"Danh sách {values}: n = {n}, nút cuối = {tail.val}")
    print(f"k = {k} -> k % n = {k % n}")
    k %= n
    if k == 0:
        print("k % n == 0 -> giữ nguyên")
        return
    tail.next = head
    print(f"Nối vòng: {tail.val} -> {head.val}")
    new_tail = tail
    for step in range(n - k):
        new_tail = new_tail.next
    print(f"Đi {n - k} bước từ nút cuối -> new_tail = {new_tail.val}, new_head = {new_tail.next.val}")
    new_head = new_tail.next
    new_tail.next = None
    print("Cắt sau new_tail ->", to_list(new_head))

trace_rotate([1, 2, 3, 4, 5], 2)
print()
trace_rotate([0, 1, 2], 4)
print()
trace_rotate([1, 2, 3], 3)

Danh sách [1, 2, 3, 4, 5]: n = 5, nút cuối = 5
k = 2 -> k % n = 2
Nối vòng: 5 -> 1
Đi 3 bước từ nút cuối -> new_tail = 3, new_head = 4
Cắt sau new_tail -> [4, 5, 1, 2, 3]

Danh sách [0, 1, 2]: n = 3, nút cuối = 2
k = 4 -> k % n = 1
Nối vòng: 2 -> 0
Đi 2 bước từ nút cuối -> new_tail = 1, new_head = 2
Cắt sau new_tail -> [2, 0, 1]

Danh sách [1, 2, 3]: n = 3, nút cuối = 3
k = 3 -> k % n = 0
k % n == 0 -> giữ nguyên


# 8. Cách 2: hai con trỏ cách nhau `k`

Sau khi tính `k %= n`, nút **mới cuối** là nút thứ `k + 1` **tính từ cuối** — đúng dạng bài #19. Cho `fast` đi trước `k` bước rồi cùng đi tới khi `fast` là nút cuối; khi đó `slow` là `new_tail`.

In [11]:
def rotate_two_pointers(head: Optional[ListNode], k: int) -> Optional[ListNode]:
    if not head or not head.next:
        return head
    n, node = 0, head
    while node:
        n += 1
        node = node.next
    k %= n
    if k == 0:
        return head
    fast = slow = head
    for _ in range(k):
        fast = fast.next
    while fast.next:
        fast = fast.next
        slow = slow.next
    new_head = slow.next
    slow.next = None
    fast.next = head
    return new_head

print(to_list(rotate_two_pointers(build_list([1, 2, 3, 4, 5]), 2)))

[4, 5, 1, 2, 3]


# 9. Phiên bản đúng format LeetCode

In [12]:
class Solution:
    def rotateRight(self, head: Optional[ListNode], k: int) -> Optional[ListNode]:
        if not head or not head.next:
            return head
        n, tail = 1, head
        while tail.next:
            tail = tail.next
            n += 1
        k %= n
        if k == 0:
            return head
        tail.next = head
        new_tail = tail
        for _ in range(n - k):
            new_tail = new_tail.next
        new_head = new_tail.next
        new_tail.next = None
        return new_head

In [13]:
def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"{name:<24} PASS  {actual}")


sol = Solution()
run = lambda vals, k: to_list(sol.rotateRight(build_list(vals), k))
check("Ví dụ 1", run([1, 2, 3, 4, 5], 2), [4, 5, 1, 2, 3])
check("Ví dụ 2", run([0, 1, 2], 4), [2, 0, 1])
check("Rỗng", run([], 5), [])
check("Một nút", run([7], 10**9), [7])
check("k = 0", run([1, 2], 0), [1, 2])
check("k = n", run([1, 2, 3], 3), [1, 2, 3])
check("k cực lớn", run([1, 2, 3], 2 * 10**9), [2, 3, 1])

import random
random.seed(61)
for _ in range(3000):
    vals = [random.randint(-100, 100) for _ in range(random.randint(0, 12))]
    k = random.randint(0, 30)
    s = k % len(vals) if vals else 0
    expected = vals[len(vals) - s:] + vals[:len(vals) - s] if vals else []
    assert run(vals, k) == expected, (vals, k)
    assert to_list(rotate_two_pointers(build_list(vals), k)) == expected
    assert to_list(rotate_step_by_step(build_list(vals), k)) == expected
print("3000 test ngẫu nhiên: ba cách khớp với cắt-ghép list Python")

Ví dụ 1                  PASS  [4, 5, 1, 2, 3]
Ví dụ 2                  PASS  [2, 0, 1]
Rỗng                     PASS  []
Một nút                  PASS  [7]
k = 0                    PASS  [1, 2]
k = n                    PASS  [1, 2, 3]
k cực lớn                PASS  [2, 3, 1]


3000 test ngẫu nhiên: ba cách khớp với cắt-ghép list Python


# 10. Vì sao đúng? Độ phức tạp?

Xoay phải `k` lần (với `0 < k < n`) đưa `k` nút cuối lên đầu, giữ nguyên thứ tự bên trong. Kết quả là `[n-k+1, …, n, 1, …, n-k]`. Lời giải tạo đúng chuỗi đó: nối `n → 1` để hai phần liền nhau, cắt sau nút `n - k`, bắt đầu từ nút `n - k + 1`.

```text
Time:  O(n)   — một lần đếm + tối đa n bước
Space: O(1)
```

So với xoay từng bước `O(k · n)` — `k` có thể tới `2 · 10^9`.

# 11. Những lỗi dễ mắc

## Lỗi 1: Không lấy `k % n`

Dù dùng cách cắt ghép, nếu đi `n - k` bước với `k > n` thì `n - k` âm → `range` rỗng → cắt sai chỗ.

In [14]:
def wrong_no_mod(head: Optional[ListNode], k: int) -> Optional[ListNode]:
    if not head or not head.next:
        return head
    n, tail = 1, head
    while tail.next:
        tail, n = tail.next, n + 1
    tail.next = head
    new_tail = tail
    for _ in range(n - k):              # SAI khi k >= n
        new_tail = new_tail.next
    new_head = new_tail.next
    new_tail.next = None
    return new_head

print("Sai :", to_list(wrong_no_mod(build_list([0, 1, 2]), 4)), "| Đúng:", run([0, 1, 2], 4))

Sai : [0, 1, 2] | Đúng: [2, 0, 1]


## Lỗi 2: Quên trường hợp `k % n == 0`

Nếu không trả sớm, `n - k = n` bước sẽ đưa `new_tail` về lại `tail` → vẫn đúng, nhưng nếu bạn quên **nối vòng trước** thì sẽ lỗi. Trả sớm cho an toàn và rõ ràng.

## Lỗi 3: Quên cắt vòng (`new_tail.next = None`)

Danh sách trở thành **vòng tròn** → mọi thao tác duyệt sau đó lặp vô hạn.

## Lỗi 4: Quên danh sách rỗng

`head = None` → `head.next` gây `AttributeError`. Kiểm tra `if not head` trước tiên.

# 12. Mẹo nhận diện pattern

- "Xoay / dịch vòng `k` lần" → **`k % n`** trước tiên (cũng áp dụng cho mảng, chuỗi).
- Linked List: muốn đưa đoạn cuối lên đầu → **nối vòng + cắt**.
- Tìm nút thứ `k` từ cuối → hai con trỏ cách `k` (**#19**).

Bài liên quan: **#189 Rotate Array**, **#796 Rotate String**, **#19 Remove Nth Node From End**, **#206 Reverse Linked List**.

# 13. Bài tập vận dụng

## Bài 1 — Tính bằng tay

Xoay `[10, 20, 30, 40]` với `k = 1`, `k = 6`, `k = 8`. Ghi `n`, `k % n`, `new_tail`, `new_head`.

## Bài 2 — Điền chỗ trống

In [15]:
def exercise_2_rotate_right(head: Optional[ListNode], k: int) -> Optional[ListNode]:
    if not head or not head.next:
        return head
    # TODO 1: đếm n, tìm tail
    # TODO 2: k %= n; nếu k == 0 trả head
    # TODO 3: nối vòng, đi n - k bước từ tail, cắt
    pass


# assert to_list(exercise_2_rotate_right(build_list([1, 2, 3, 4, 5]), 2)) == [4, 5, 1, 2, 3]
# assert to_list(exercise_2_rotate_right(build_list([0, 1, 2]), 4)) == [2, 0, 1]
# assert to_list(exercise_2_rotate_right(build_list([1, 2]), 2)) == [1, 2]

## Bài 3 — Biến thể: xoay sang **trái**

Xoay trái `k` vị trí (nút đầu chuyển xuống cuối).

```python
rotate_left([1, 2, 3, 4, 5], 2) -> [3, 4, 5, 1, 2]
```

In [16]:
def exercise_3_rotate_left(head: Optional[ListNode], k: int) -> Optional[ListNode]:
    # TODO
    pass


# assert to_list(exercise_3_rotate_left(build_list([1, 2, 3, 4, 5]), 2)) == [3, 4, 5, 1, 2]
# assert to_list(exercise_3_rotate_left(build_list([1, 2, 3]), 7)) == [2, 3, 1]
# assert to_list(exercise_3_rotate_left(build_list([]), 1)) == []

## Bài 4 — Biến thể: Rotate Array (LeetCode #189)

Xoay mảng sang phải `k` bước **tại chỗ**, bộ nhớ `O(1)`.

Gợi ý nhanh: **đảo cả mảng**, rồi đảo `k` phần tử đầu, rồi đảo phần còn lại.

```python
a = [1, 2, 3, 4, 5, 6, 7]; rotate_array(a, 3)  ->  a == [5, 6, 7, 1, 2, 3, 4]
```

In [17]:
def exercise_4_rotate_array(nums: List[int], k: int) -> None:
    # TODO
    pass


# a = [1, 2, 3, 4, 5, 6, 7]; exercise_4_rotate_array(a, 3); assert a == [5, 6, 7, 1, 2, 3, 4]
# a = [-1, -100, 3, 99]; exercise_4_rotate_array(a, 2); assert a == [3, 99, -1, -100]
# a = [1]; exercise_4_rotate_array(a, 5); assert a == [1]

## Bài 5 — Mini challenge: Rotate String (LeetCode #796)

`goal` có phải là một phép xoay của `s` không?

```python
rotate_string("abcde", "cdeab") -> True
rotate_string("abcde", "abced") -> False
```

Gợi ý nhanh: mọi phép xoay của `s` đều là chuỗi con của `s + s`.

In [18]:
def exercise_5_rotate_string(s: str, goal: str) -> bool:
    # TODO
    pass


# assert exercise_5_rotate_string("abcde", "cdeab") is True
# assert exercise_5_rotate_string("abcde", "abced") is False
# assert exercise_5_rotate_string("a", "aa") is False

# 14. Gợi ý

**Bài 1**
- Mức 1: `n = 4`; `6 % 4 = 2`; `8 % 4 = 0`.

**Bài 3**
- Mức 1: xoay trái `k` = xoay phải `n - (k % n)`.

**Bài 4**
- Mức 1: `k %= n` trước.
- Mức 2: viết hàm `reverse(lo, hi)` đổi chỗ hai đầu tiến vào giữa.

**Bài 5**
- Mức 1: `len(s) == len(goal) and goal in s + s`.

# 15. Đáp án tham khảo

Bạn nên tự làm trước khi xem phần này.

---

## Đáp án Bài 1

In [19]:
for k in [1, 6, 8]:
    trace_rotate([10, 20, 30, 40], k)
    print()

Danh sách [10, 20, 30, 40]: n = 4, nút cuối = 40
k = 1 -> k % n = 1
Nối vòng: 40 -> 10
Đi 3 bước từ nút cuối -> new_tail = 30, new_head = 40
Cắt sau new_tail -> [40, 10, 20, 30]

Danh sách [10, 20, 30, 40]: n = 4, nút cuối = 40
k = 6 -> k % n = 2
Nối vòng: 40 -> 10
Đi 2 bước từ nút cuối -> new_tail = 20, new_head = 30
Cắt sau new_tail -> [30, 40, 10, 20]

Danh sách [10, 20, 30, 40]: n = 4, nút cuối = 40
k = 8 -> k % n = 0
k % n == 0 -> giữ nguyên



## Đáp án Bài 2

In [20]:
def answer_2_rotate_right(head: Optional[ListNode], k: int) -> Optional[ListNode]:
    return Solution().rotateRight(head, k)

assert to_list(answer_2_rotate_right(build_list([1, 2, 3, 4, 5]), 2)) == [4, 5, 1, 2, 3]
assert to_list(answer_2_rotate_right(build_list([0, 1, 2]), 4)) == [2, 0, 1]
assert to_list(answer_2_rotate_right(build_list([1, 2]), 2)) == [1, 2]
print("Bài 2 PASS (giống lời giải chính mục 9)")

Bài 2 PASS (giống lời giải chính mục 9)


## Đáp án Bài 3

In [21]:
def answer_3_rotate_left(head: Optional[ListNode], k: int) -> Optional[ListNode]:
    if not head:
        return head
    n, node = 0, head
    while node:
        n, node = n + 1, node.next
    return Solution().rotateRight(head, (n - k % n) % n)

assert to_list(answer_3_rotate_left(build_list([1, 2, 3, 4, 5]), 2)) == [3, 4, 5, 1, 2]
assert to_list(answer_3_rotate_left(build_list([1, 2, 3]), 7)) == [2, 3, 1]
assert to_list(answer_3_rotate_left(build_list([]), 1)) == []
print("Bài 3 PASS")

Bài 3 PASS


## Đáp án Bài 4

In [22]:
def answer_4_rotate_array(nums: List[int], k: int) -> None:
    n = len(nums)
    k %= n

    def reverse(lo, hi):
        while lo < hi:
            nums[lo], nums[hi] = nums[hi], nums[lo]
            lo, hi = lo + 1, hi - 1

    reverse(0, n - 1)
    reverse(0, k - 1)
    reverse(k, n - 1)

a = [1, 2, 3, 4, 5, 6, 7]; answer_4_rotate_array(a, 3); assert a == [5, 6, 7, 1, 2, 3, 4]
a = [-1, -100, 3, 99]; answer_4_rotate_array(a, 2); assert a == [3, 99, -1, -100]
a = [1]; answer_4_rotate_array(a, 5); assert a == [1]
print("Bài 4 PASS")

Bài 4 PASS


`[1..7]` → đảo cả: `[7,6,5,4,3,2,1]` → đảo 3 đầu: `[5,6,7,4,3,2,1]` → đảo phần còn lại: `[5,6,7,1,2,3,4]`.

## Đáp án Bài 5

In [23]:
def answer_5_rotate_string(s: str, goal: str) -> bool:
    return len(s) == len(goal) and goal in s + s

assert answer_5_rotate_string("abcde", "cdeab") is True
assert answer_5_rotate_string("abcde", "abced") is False
assert answer_5_rotate_string("a", "aa") is False
print("Bài 5 PASS")

Bài 5 PASS


# 16. Tổng kết

> `k %= n` trước tiên. Rồi **nối vòng** (`tail.next = head`), đi `n - k` bước tới `new_tail`, **cắt** sau nó. Chỉ sửa hai mũi tên, `O(n)`.

---

## Checklist trước khi nộp

- [ ] Xử lý danh sách rỗng / 1 nút?
- [ ] `k %= n` và trả sớm khi `k == 0`?
- [ ] Nhớ cắt vòng (`new_tail.next = None`)?
- [ ] Đã thử `k = n`, `k` cực lớn?